# Base vs Fine-Tuned Evaluation
Same prompt + same decoding settings for fair comparison. Supports `hf` and `ollama` inference backends.

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml

ROOT = Path('..').resolve()
CONFIG = yaml.safe_load((ROOT / 'config.yaml').read_text())
seed = CONFIG['seed']
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)


In [ ]:
def read_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

test_rows = read_jsonl(ROOT / CONFIG['dataset']['test_path'])
print('test samples:', len(test_rows))


In [ ]:
def build_prompt(item):
    return (
        '<|im_start|>system\nYou are a strict JSON generator. Return only JSON.\n<|im_end|>\n'
        f"<|im_start|>user\nInstruction: {item['instruction']}\nInput: {item['input']}\n<|im_end|>\n"
        '<|im_start|>assistant\n'
    )

backend = CONFIG['inference'].get('backend', 'hf')
print('Inference backend:', backend)


In [ ]:
if backend == 'hf':
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    model_name = CONFIG['model_name']
    revision = CONFIG.get('base_model_revision', 'main')
    adapter_path = ROOT / CONFIG['training']['output_dir']
    if not adapter_path.exists():
        raise FileNotFoundError(
            f'Adapter directory not found: {adapter_path}. '
            'Run fine_tuning.ipynb on a CUDA-enabled machine first.'
        )

    use_bnb_4bit = torch.cuda.is_available()
    if not use_bnb_4bit:
        print('CUDA is not available. Falling back to non-quantized CPU inference (slow).')
        print("Tip: set `inference.backend: ollama` in config.yaml for local inference without CUDA.")

    bnb = None
    if use_bnb_4bit:
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )

    tokenizer = AutoTokenizer.from_pretrained(
        model_name, revision=revision, trust_remote_code=True
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        revision=revision,
        quantization_config=bnb if use_bnb_4bit else None,
        device_map='auto' if use_bnb_4bit else 'cpu',
        trust_remote_code=True,
    )

    tuned_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        revision=revision,
        quantization_config=bnb if use_bnb_4bit else None,
        device_map='auto' if use_bnb_4bit else 'cpu',
        trust_remote_code=True,
    )
    tuned_model = PeftModel.from_pretrained(tuned_model, adapter_path)

    GEN_CFG = {
        'max_new_tokens': CONFIG['inference']['max_new_tokens'],
        'temperature': CONFIG['inference']['temperature'],
        'top_p': CONFIG['inference']['top_p'],
        'do_sample': CONFIG['inference']['do_sample'],
    }

    def generate_base(item):
        prompt = build_prompt(item)
        inputs = tokenizer(prompt, return_tensors='pt').to(base_model.device)
        with torch.no_grad():
            out = base_model.generate(**inputs, **GEN_CFG)
        gen_tokens = out[0][inputs['input_ids'].shape[1]:]
        return tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

    def generate_tuned(item):
        prompt = build_prompt(item)
        inputs = tokenizer(prompt, return_tensors='pt').to(tuned_model.device)
        with torch.no_grad():
            out = tuned_model.generate(**inputs, **GEN_CFG)
        gen_tokens = out[0][inputs['input_ids'].shape[1]:]
        return tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()


In [ ]:
if backend == 'ollama':
    import ollama

    # In ollama mode, base model is local ollama model.
    # tuned model should be exposed as another local ollama model tag if available.
    base_ollama_model = CONFIG['inference'].get('ollama_model', 'qwen2.5:1.5b-instruct')
    tuned_ollama_model = CONFIG['inference'].get('ollama_tuned_model', base_ollama_model)

    def _chat(model_name, item):
        prompt = build_prompt(item)
        resp = ollama.chat(
            model=model_name,
            messages=[{'role': 'user', 'content': prompt}],
            options={
                'temperature': CONFIG['inference']['temperature'],
                'top_p': CONFIG['inference']['top_p'],
                'num_predict': CONFIG['inference']['max_new_tokens'],
            },
            stream=False,
        )
        return resp['message']['content'].strip()

    def generate_base(item):
        return _chat(base_ollama_model, item)

    def generate_tuned(item):
        return _chat(tuned_ollama_model, item)


In [ ]:
def parse_json(s):
    text = s.strip()
    if '```' in text:
        text = text.replace('```json', '```').replace('```JSON', '```')
        parts = [p.strip() for p in text.split('```') if p.strip()]
        for p in parts:
            if p.startswith('{') and p.endswith('}'):
                text = p
                break

    start = text.find('{')
    end = text.rfind('}')
    if start != -1 and end != -1 and end > start:
        text = text[start:end + 1]

    try:
        return json.loads(text), True
    except Exception:
        return None, False

rows = []
for ex in test_rows:
    expected = json.loads(ex['output'])
    b_raw = generate_base(ex)
    t_raw = generate_tuned(ex)

    b_obj, b_valid = parse_json(b_raw)
    t_obj, t_valid = parse_json(t_raw)

    def exact(obj):
        return int(obj == expected) if obj is not None else 0

    def field_acc(obj):
        if obj is None:
            return 0.0
        keys = ['task_type', 'label', 'confidence', 'evidence']
        return float(sum(1 for k in keys if obj.get(k) == expected.get(k)) / len(keys))

    rows.append({
        'input': ex['input'],
        'expected': expected,
        'base_raw': b_raw,
        'tuned_raw': t_raw,
        'base_output': b_obj,
        'tuned_output': t_obj,
        'base_valid_json': b_valid,
        'tuned_valid_json': t_valid,
        'base_exact': exact(b_obj),
        'tuned_exact': exact(t_obj),
        'base_field_acc': field_acc(b_obj),
        'tuned_field_acc': field_acc(t_obj),
    })

df = pd.DataFrame(rows)
metrics = pd.DataFrame([
    {
        'metric': 'exact_match',
        'base': df['base_exact'].mean(),
        'fine_tuned': df['tuned_exact'].mean(),
    },
    {
        'metric': 'field_level_accuracy',
        'base': df['base_field_acc'].mean(),
        'fine_tuned': df['tuned_field_acc'].mean(),
    },
    {
        'metric': 'json_validity',
        'base': df['base_valid_json'].mean(),
        'fine_tuned': df['tuned_valid_json'].mean(),
    },
])

display(metrics)


In [ ]:
def make_comment(row):
    if row['tuned_exact'] > row['base_exact']:
        return 'fine-tuned reached exact match'
    if row['tuned_field_acc'] > row['base_field_acc']:
        return 'fine-tuned improved field-level match'
    if row['tuned_valid_json'] and not row['base_valid_json']:
        return 'fine-tuned improved JSON validity'
    return 'no improvement'

improved = df[(df['tuned_exact'] > df['base_exact']) | (df['tuned_field_acc'] > df['base_field_acc'])].head(5).copy()
errors = df[(df['tuned_exact'] == 0)].head(5).copy()

improved['comment'] = improved.apply(make_comment, axis=1)
errors['comment'] = 'fine-tuned output does not exactly match expected JSON'

print('Improved cases:', len(improved))
display(improved[['input', 'expected', 'base_output', 'tuned_output', 'comment']])

print('Fine-tuned error cases:', len(errors))
display(errors[['input', 'expected', 'base_output', 'tuned_output', 'comment']])
